In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import sys
from pathlib import Path

# Go to project root (REALTIMEFRAUDDETECTION)
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [4]:
df=pd.read_csv("../data/dataset_PFE_CDM_complet.csv")
df.shape

(1100000, 17)

In [5]:
df['Time']=pd.to_datetime(df['Time'])
df['Age']=df['Age'].astype(int)

df['Hour']=df['Time'].dt.hour
df['Date']=df['Time'].dt.date

In [6]:
df.sort_values(['Account Number','Time'],inplace=True)
df['rank']=df.groupby("Account Number")['Time'].rank(method='first',pct=True)

# Feature Engineering

In [7]:
from src.features.TimeFeatures import ComputeTimeFeatures
from src.features.CatEntropy import ComputeCatEntropy
from src.features.CatFreq import ComputeCatFreq
from src.preprocessing.TrainTestSplit import split

In [8]:
df=ComputeTimeFeatures(df)
df=ComputeCatEntropy(df)
df=ComputeCatFreq(df)

In [9]:
df=df[['Age','rank',
       'LogAmount', 'MovingAvg','MovingStd', 'LogTimeDiff','Hour',
       'TransactionTypeEntropy', 'ChannelEntropy','CardTypeEntropy', 'MerchandEntropy', 'CountryEntropy', 'CityEntropy',
       'TransactionTypeFreq','ChannelFreq','CardTypeFreq', 'MerchandFreq', 'CountryFreq','CityFreq']]

traindf,testdf=split(df)

In [10]:
traindf.shape

(879198, 18)

# Isolation Forest

In [11]:
from src.models.IsolationForest import IsolationForestModel
from src.training.train import trainmodel

2026-03-04 11:19:45.794314: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-04 11:19:45.842862: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 11:19:47.445390: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [12]:
# iso=IsolationForestModel()

# trainmodel(iso,traindf,runname='testIF')

In [13]:
def ComputeAvgPathLength(model, X):
    depths = []

    for tree in model.estimators_:
        node_indicator = tree.decision_path(X)
        depth = node_indicator.sum(axis=1)
        depths.append(np.array(depth).reshape(-1))

    depths = np.array(depths)
    return depths.mean(axis=0)

# AutoEncoder

In [14]:
from src.models.AutoEncoder import AutoEncoderModel

In [15]:
inputdim=traindf.shape[1]

ae=AutoEncoderModel(inputdim=inputdim,epochs=10)

trainmodel(ae,traindf,runname='testAE')

2026-03-04 11:19:55.336209: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - loss: 0.4084 - val_loss: 0.1391
Epoch 2/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - loss: 0.0862 - val_loss: 0.0615
Epoch 3/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0529 - val_loss: 0.0539
Epoch 4/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - loss: 0.0494 - val_loss: 0.0549
Epoch 5/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0250 - val_loss: 0.0223
Epoch 6/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0216 - val_loss: 0.0211
Epoch 7/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0215 - val_loss: 0.0209
Epoch 8/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0213 - val_loss: 0.0207
Epoch 9/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0212 - val_loss: 0.0208
Epoch 10/10
12364/12364 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - loss: 0.0212 - val_loss: 0.0208
27475/27475 ━━━━━━━━━━━━━━━━━━━━ 29s 1ms/step


2026/03/04 11:25:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 11:25:10 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


{'score_mean': 0.020830675284712323,
 'score_std': 0.01813669520259553,
 'score_p95': 0.05418210112286649}

# SelfOrganizingMap

In [15]:
from src.models.SelfOrganizingMap import SOMModel

In [ ]:
# model=SOMModel(inputlen=inputdim)

# trainmodel(model,traindf,runname='testSOM')

{'score_mean': 4.159387083719637,
 'score_std': 2.25767770545966,
 'score_p95': 9.047119070413597}

# HyperParam Tuning

In [16]:
from src.training.tune import tunemodel
from src.models.AutoEncoder import AutoEncoderModel
from src.training.seachspaces import aesearchspace

In [17]:
study = tunemodel(
    AutoEncoderModel,
    aesearchspace,
    traindf,
    ntrials=30
)

[I 2026-03-04 11:25:59,917] A new study created in memory with name: no-name-86b146ab-7870-434a-a67b-2ed9b8249f16


Epoch 1/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5709 - val_loss: 0.0096
Epoch 2/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0085 - val_loss: 0.0023
Epoch 3/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0043 - val_loss: 0.0030
Epoch 4/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0038 - val_loss: 0.0022
Epoch 5/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0035 - val_loss: 0.0011
Epoch 6/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0034 - val_loss: 0.0017
Epoch 7/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0031 - val_loss: 9.3669e-04
Epoch 8/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0030 - val_loss: 4.7804e-04
Epoch 9/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0027 - val_loss: 6.4250e-04
Epoch 10/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0026 - val_loss: 6.1834e-04
Epoch 11/50
3091/3091 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0027 - val_loss: 4.6851e-04
Epoch 12/

[I 2026-03-04 11:31:53,959] Trial 0 finished with value: 0.0002810369721570437 and parameters: {'latent_dim': 24, 'lr': 0.0018869104067433106, 'batch_size': 256}. Best is trial 0 with value: 0.0002810369721570437.


Epoch 1/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 0.3166 - val_loss: 0.0074
Epoch 2/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0085 - val_loss: 0.0044
Epoch 3/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0058 - val_loss: 0.0015
Epoch 4/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 0.0045 - val_loss: 0.0025
Epoch 5/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 0.0042 - val_loss: 0.0012
Epoch 6/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0039 - val_loss: 0.0290
Epoch 7/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 0.0035 - val_loss: 0.0038
Epoch 8/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0033 - val_loss: 0.0010
Epoch 9/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0030 - val_loss: 0.0032
Epoch 10/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0029 - val_loss: 6.7280e-04
Epoch 11/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0027 - val_loss: 0.0011
Epoch 12/50
618

[I 2026-03-04 11:39:54,191] Trial 1 finished with value: 0.00046363729040758663 and parameters: {'latent_dim': 22, 'lr': 0.0025686430507167774, 'batch_size': 128}. Best is trial 0 with value: 0.0002810369721570437.


Epoch 1/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - loss: 0.4854 - val_loss: 0.1429
Epoch 2/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.1219 - val_loss: 0.1130
Epoch 3/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0997 - val_loss: 0.0999
Epoch 4/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0927 - val_loss: 0.0909
Epoch 5/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0886 - val_loss: 0.0880
Epoch 6/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0847 - val_loss: 0.0864
Epoch 7/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0829 - val_loss: 0.0835
Epoch 8/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0795 - val_loss: 0.0675
Epoch 9/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0645 - val_loss: 0.0616
Epoch 10/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0601 - val_loss: 0.0574
Epoch 11/50
6182/6182 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0592 - val_loss: 0.0559
Epoch 12/50
6182/61

[I 2026-03-04 11:53:27,134] Trial 2 finished with value: 0.01993192352564044 and parameters: {'latent_dim': 6, 'lr': 0.0015091882778570718, 'batch_size': 128}. Best is trial 0 with value: 0.0002810369721570437.


Epoch 1/50
  987/12364 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - loss: 5.9695

[W 2026-03-04 11:53:31,325] Trial 3 failed with parameters: {'latent_dim': 15, 'lr': 0.007086779668699319, 'batch_size': 64} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/aymanfen/Desktop/RealTimeFraudDetection/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/home/aymanfen/Desktop/RealTimeFraudDetection/src/training/tune.py", line 14, in objective
    model.fit(Xtrain)
    ~~~~~~~~~^^^^^^^^
  File "/home/aymanfen/Desktop/RealTimeFraudDetection/src/models/AutoEncoder.py", line 50, in fit
    history=self.model.fit(X,X,
                   epochs=self.epochs,batch_size=self.batch_size,
                   validation_split=self.validation_split,callbacks=callbacks,
                   shuffle=True)
  File "/home/aymanfen/Desktop/RealTimeFraudDetection/.venv/lib/python3.13/site-packages/keras/src/utils/traceback_utils.py", line 117, in error_handler
    retu

KeyboardInterrupt: 

In [22]:
print(study.best_params)

{'x': 15, 'y': 19, 'sigma': 0.5059489302795622, 'learning_rate': 0.27377346330062596, 'iterations': 826}
